In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

In [3]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [4]:
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

As of April 2026, several AI agent frameworks have emerged, each catering to specific use cases:

- **LangGraph**: Ideal for complex, stateful workflows, LangGraph models agent behavior as directed graphs, supporting single, multi, and hierarchical-agent patterns. ([workday.com](https://www.workday.com/en-us/perspectives/ai/2026/01/top-ai-agent-frameworks.html?utm_source=openai))

- **CrewAI**: Designed for role-based agent teams, CrewAI facilitates collaboration among specialized agents, making it suitable for research teams and content generation. ([leaper.dev](https://leaper.dev/blog/ai-agent-frameworks-2026?utm_source=openai))

- **AutoGen**: Developed by Microsoft Research, AutoGen focuses on conversational multi-agent systems, enabling agents to engage in complex dialogues and task coordination. ([delx.ai](https://delx.ai/agents/best-ai-agent-frameworks-2026?utm_source=openai))

- **OpenClaw**: Emphasizing interactive assistants and team integration, OpenClaw offers channel-native communication and a flexible skill system, supporting deployment across multiple devices. ([openclawsetup.dev](https://openclawsetup.dev/blog/best-ai-agent-frameworks-2026?utm_source=openai))

- **Semantic Kernel**: Backed by Microsoft, Semantic Kernel is tailored for enterprise .NET/Java agents, providing a robust framework for integrating AI capabilities into existing enterprise applications. ([delx.ai](https://delx.ai/agents/best-ai-agent-frameworks-2026?utm_source=openai))

The AI agent framework ecosystem is consolidating around these major players, with the OpenAI Agents SDK rapidly gaining popularity. This consolidation reflects a maturation in the field, with developers gravitating towards frameworks that offer reliability, scalability, and ease of integration. ([agentscout.live](https://agentscout.live/tech/ai-agents/insight/ai-agent-framework-consolidation-developer-preference-shifts/?utm_source=openai))

In addition to these frameworks, companies like Salesforce and Okta are advancing the development and security of AI agents. Salesforce's AI Foundry project focuses on bridging the gap between research and product innovation, while Okta's "secure agentic enterprise" framework aims to enhance the management and security of AI agents within organizations. ([itpro.com](https://www.itpro.com/technology/artificial-intelligence/salesforce-ramps-up-agentic-ai-research-with-new-foundry-project?utm_source=openai))

Furthermore, Nvidia's formation of the Nemotron Coalition, comprising eight AI companies, signifies a collaborative effort to develop open frontier models, promoting interoperability and innovation in the AI agent space. ([tomshardware.com](https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-nemoclaw-coalition-brings-eight-ai-labs-together-to-build-open-frontier-models?utm_source=openai))


## Highlights:
- [Salesforce ramps up agentic AI research with new foundry project](https://www.itpro.com/technology/artificial-intelligence/salesforce-ramps-up-agentic-ai-research-with-new-foundry-project?utm_source=openai), Published on Friday, March 27
- [Okta unveils new framework to secure and protect enterprise AI agents](https://www.techradar.com/pro/security/okta-unveils-new-framework-to-secure-and-protect-enterprise-ai-agents?utm_source=openai), Published on Tuesday, March 17
- [Nvidia's Nemotron coalition brings eight AI labs together to build open frontier models](https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-nemoclaw-coalition-brings-eight-ai-labs-together-to-build-open-frontier-models?utm_source=openai), Published on Monday, March 16 

In [5]:

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)